Script used to analyse the data files found here: [ClimSim_low-res](https://huggingface.co/datasets/LEAP/ClimSim_low-res/tree/main)


In [3]:
import os
import xarray as xr
import tqdm

In [4]:
main_folder_path = '/gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/'
target_file_path = os.path.join(main_folder_path, '0001-02', 'E3SM-MMF.mli.0001-02-17-39600.nc')

# Exploring the data

In [10]:
!ls $main_folder_path/0001-02/

E3SM-MMF.mli.0001-02-01-00000.nc  E3SM-MMF.mlo.0001-02-01-00000.nc
E3SM-MMF.mli.0001-02-01-01200.nc  E3SM-MMF.mlo.0001-02-01-01200.nc
E3SM-MMF.mli.0001-02-01-02400.nc  E3SM-MMF.mlo.0001-02-01-02400.nc
E3SM-MMF.mli.0001-02-01-03600.nc  E3SM-MMF.mlo.0001-02-01-03600.nc
E3SM-MMF.mli.0001-02-01-04800.nc  E3SM-MMF.mlo.0001-02-01-04800.nc
E3SM-MMF.mli.0001-02-01-06000.nc  E3SM-MMF.mlo.0001-02-01-06000.nc
E3SM-MMF.mli.0001-02-01-07200.nc  E3SM-MMF.mlo.0001-02-01-07200.nc
E3SM-MMF.mli.0001-02-01-08400.nc  E3SM-MMF.mlo.0001-02-01-08400.nc
E3SM-MMF.mli.0001-02-01-09600.nc  E3SM-MMF.mlo.0001-02-01-09600.nc
E3SM-MMF.mli.0001-02-01-10800.nc  E3SM-MMF.mlo.0001-02-01-10800.nc
E3SM-MMF.mli.0001-02-01-12000.nc  E3SM-MMF.mlo.0001-02-01-12000.nc
E3SM-MMF.mli.0001-02-01-13200.nc  E3SM-MMF.mlo.0001-02-01-13200.nc
E3SM-MMF.mli.0001-02-01-14400.nc  E3SM-MMF.mlo.0001-02-01-14400.nc
E3SM-MMF.mli.0001-02-01-15600.nc  E3SM-MMF.mlo.0001-02-01-15600.nc
E3SM-MMF.mli.0001-02-01-16800.nc  E3SM-MMF.mlo.0001-02-01-1680

In [5]:
v1_inputs = ['state_t',
                          'state_q0001',
                          'state_ps',
                          'pbuf_SOLIN',
                          'pbuf_LHFLX',
                          'pbuf_SHFLX']

In [7]:
# Open the netCDF file 
ds = xr.open_dataset(target_file_path)
ds

<xarray.Dataset> Size: 2MB
Dimensions:           (ncol: 384, lev: 60)
Dimensions without coordinates: ncol, lev
Data variables: (12/29)
    ymd               int32 4B ...
    tod               int32 4B ...
    cam_in_ALDIF      (ncol) float64 3kB ...
    cam_in_ALDIR      (ncol) float64 3kB ...
    cam_in_ASDIF      (ncol) float64 3kB ...
    cam_in_ASDIR      (ncol) float64 3kB ...
    ...                ...
    state_t           (lev, ncol) float64 184kB ...
    state_u           (lev, ncol) float64 184kB ...
    state_v           (lev, ncol) float64 184kB ...
    pbuf_CH4          (lev, ncol) float64 184kB ...
    pbuf_N2O          (lev, ncol) float64 184kB ...
    pbuf_ozone        (lev, ncol) float64 184kB ...
Attributes:
    ne:        4
    fv_nphys:  2
    calendar:  NO_LEAP

In [8]:
# Create a new dataset with only the specified variables
ds_new = ds[v1_inputs]
ds_new

<xarray.Dataset> Size: 381kB
Dimensions:      (lev: 60, ncol: 384)
Dimensions without coordinates: lev, ncol
Data variables:
    state_t      (lev, ncol) float64 184kB ...
    state_q0001  (lev, ncol) float64 184kB ...
    state_ps     (ncol) float64 3kB ...
    pbuf_SOLIN   (ncol) float64 3kB ...
    pbuf_LHFLX   (ncol) float64 3kB ...
    pbuf_SHFLX   (ncol) float64 3kB ...
Attributes:
    ne:        4
    fv_nphys:  2
    calendar:  NO_LEAP

# Dummy loading

In [44]:
target_years = ['0001']
target_months = ['02', '03']

filelist = []
for year in target_years:
    for month in target_months:
        folder_path = os.path.join(main_folder_path, f"{year}-{month}")
        for file_name in os.listdir(folder_path):
            if file_name.endswith('.nc'):
                filelist.append(os.path.join(folder_path, file_name))

stride_sample = 100
start_idx = 0
end_idx = -1
print(len(filelist))
print(sorted(filelist))
sampled_datalist = sorted(filelist)[start_idx:end_idx:stride_sample]
print(len(sampled_datalist))
print(sorted(sampled_datalist))

new_dataset_list = []
missing_vars = {x:0 for x in v1_inputs}
for file_path in tqdm.tqdm(sampled_datalist):
    ds = xr.open_dataset(file_path)
    try:
        new_dataset_list.append(ds[v1_inputs])
    except:
        for var in v1_inputs:
            if var not in ds:
                missing_vars[var] += 1
        
        print(f"Warning error getting data from {file_path}, skipping drop.")

    # print(f"Opened {file_path}")
# Combine all datasets into one
print(missing_vars)
combined_ds = xr.concat(new_dataset_list, dim='time')
combined_ds




#                 file_path = os.path.join(folder_path, file_name)
#                 ds = xr.open_dataset(file_path)
#                 new_dataset_list.append(ds[v1_inputs])
#                 print(f"Opened {file_path} with variables: {list(ds.data_vars)}")
# # Combine all datasets into one
# combined_ds = xr.concat(new_dataset_list, dim='time')
# combined_ds

8496
['/gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mli.0001-02-01-00000.nc', '/gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mli.0001-02-01-01200.nc', '/gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mli.0001-02-01-02400.nc', '/gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mli.0001-02-01-03600.nc', '/gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mli.0001-02-01-04800.nc', '/gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mli.0001-02-01-06000.nc', '/gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mli.0001-02-01-07200.nc', '/gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mli.0001-02-01-08400.nc', '/gws/nopw/j04/iecdt/bstanleyclamp/subsamp

 28%|██▊       | 24/85 [00:01<00:02, 22.17it/s]

Warning error getting data from /gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mlo.0001-02-02-14400.nc, skipping drop.
Warning error getting data from /gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mlo.0001-02-03-48000.nc, skipping drop.
Warning error getting data from /gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mlo.0001-02-04-81600.nc, skipping drop.
Warning error getting data from /gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mlo.0001-02-06-28800.nc, skipping drop.
Warning error getting data from /gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mlo.0001-02-07-62400.nc, skipping drop.


 35%|███▌      | 30/85 [00:01<00:02, 23.43it/s]

Warning error getting data from /gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mlo.0001-02-09-09600.nc, skipping drop.
Warning error getting data from /gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mlo.0001-02-10-43200.nc, skipping drop.
Warning error getting data from /gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mlo.0001-02-11-76800.nc, skipping drop.
Warning error getting data from /gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mlo.0001-02-13-24000.nc, skipping drop.
Warning error getting data from /gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mlo.0001-02-14-57600.nc, skipping drop.
Warning error getting data from /gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mlo.0001-02-16-04800.nc, skipping drop.


...

 95%|█████████▌| 81/85 [00:03<00:00, 20.92it/s]

Warning error getting data from /gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-03/E3SM-MMF.mlo.0001-03-19-48000.nc, skipping drop.
Warning error getting data from /gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-03/E3SM-MMF.mlo.0001-03-20-81600.nc, skipping drop.
Warning error getting data from /gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-03/E3SM-MMF.mlo.0001-03-22-28800.nc, skipping drop.
Warning error getting data from /gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-03/E3SM-MMF.mlo.0001-03-23-62400.nc, skipping drop.
Warning error getting data from /gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-03/E3SM-MMF.mlo.0001-03-25-09600.nc, skipping drop.
Warning error getting data from /gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-03/E3SM-MMF.mlo.0001-03-26-43200.nc, skipping drop.


100%|██████████| 85/85 [00:04<00:00, 20.88it/s]


Warning error getting data from /gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-03/E3SM-MMF.mlo.0001-03-27-76800.nc, skipping drop.
Warning error getting data from /gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-03/E3SM-MMF.mlo.0001-03-29-24000.nc, skipping drop.
Warning error getting data from /gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-03/E3SM-MMF.mlo.0001-03-30-57600.nc, skipping drop.
{'state_t': 0, 'state_q0001': 0, 'state_ps': 42, 'pbuf_SOLIN': 42, 'pbuf_LHFLX': 42, 'pbuf_SHFLX': 42}


<xarray.Dataset> Size: 16MB
Dimensions:      (time: 43, lev: 60, ncol: 384)
Dimensions without coordinates: time, lev, ncol
Data variables:
    state_t      (time, lev, ncol) float64 8MB 213.8 213.2 217.1 ... 276.3 288.4
    state_q0001  (time, lev, ncol) float64 8MB 1.485e-06 1.487e-06 ... 0.01055
    state_ps     (time, ncol) float64 132kB 1.011e+05 1.014e+05 ... 9.807e+04
    pbuf_SOLIN   (time, ncol) float64 132kB 0.0 0.0 0.0 ... 1.008e+03 1.093e+03
    pbuf_LHFLX   (time, ncol) float64 132kB 89.49 102.3 187.1 ... -0.1797 11.38
    pbuf_SHFLX   (time, ncol) float64 132kB 5.66 10.92 6.692 ... 0.3031 -5.117
Attributes:
    ne:        4
    fv_nphys:  2
    calendar:  NO_LEAP

In [ ]:
def get_dataset_filenames(base_folder_path: str, target_years: list, target_months: list, input_regex: str='E3SM-MMF.mli', target_regex: str='E3SM-MMF.mlo') -> list:
        """
        Extracts filenames for the dataset, based on requirements given in the dataset config, ensuring both input and target files are found.

        Args:
            base_folder_path: Base folder path where raw data files are stored.
            target_years: List of years to include in the dataset.
            target_months: List of months to include in the dataset.
            input_regex: Regex pattern to identify input files.
            target_regex: Regex pattern to identify target files.

        Returns:
            List (sorted) of filenames for the dataset.
        """
        input_filelist = []
        target_filelist = []
        for year in target_years:
            for month in target_months:
                folder_path = os.path.join(base_folder_path, f"{year}-{month}")
                for filename in os.listdir(folder_path):
                    if input_regex in filename:
                        input_filelist.append(os.path.join(folder_path, filename))
                    elif target_regex in filename:
                        target_filelist.append(os.path.join(folder_path, filename))
        
        return sorted(input_filelist), sorted(target_filelist)

In [11]:
def check_matching_files(input_files, target_files):
    """
    Checks if every input file has a corresponding target file 
    by comparing their normalized filenames (excluding the .mli./.mlo. part).
    """

    # 1. Define the normalization function (removes the file type marker)
    # This function replaces '.mli.' and '.mlo.' with just '.' 
    # to create a common, comparable string (the unique ID).
    normalize = lambda s: s.replace('.mli.', '.').replace('.mlo.', '.')
    
    # 2. Normalize both lists
    normalized_inputs = [normalize(f) for f in input_files]
    normalized_targets = [normalize(f) for f in target_files]
    
    # 3. Check for matching length and content
    # They must have the same length AND the sorted normalized lists must be identical.
    if len(normalized_inputs) != len(normalized_targets):
        print(f"Mismatch: Input list has {len(input_files)} files, Target has {len(target_files)}.")
        return False
        
    # Check if all normalized file names are an exact match (requires sorting)
    return sorted(normalized_inputs) == sorted(normalized_targets)

In [13]:
target_years = ['0001']
target_months = ['02', '03']
input_files, target_files = get_dataset_filenames(main_folder_path, target_years, target_months)
print("Input files:")
print(input_files)
print("Target files:")
print(target_files)
print("Do input and target files match?", check_matching_files(input_files[:-1], target_files))

Input files:
['/gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mli.0001-02-01-00000.nc', '/gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mli.0001-02-01-01200.nc', '/gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mli.0001-02-01-02400.nc', '/gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mli.0001-02-01-03600.nc', '/gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mli.0001-02-01-04800.nc', '/gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mli.0001-02-01-06000.nc', '/gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mli.0001-02-01-07200.nc', '/gws/nopw/j04/iecdt/bstanleyclamp/subsampled_low_res/ClimSim_low-res/train/0001-02/E3SM-MMF.mli.0001-02-01-08400.nc', '/gws/nopw/j04/iecdt/bstanleyclamp

In [27]:
print(len(new_dataset_list))
print(len(combined_ds['time']))
print(combined_ds.data_vars['state_t'].shape)
print(combined_ds.data_vars['state_ps'].shape)



43
43
(43, 60, 384)
(43, 384)


# Now lets look at normalisation 

In [42]:
normalised_stats = {}
normalised_ds = combined_ds.copy()
for var in combined_ds.data_vars:
    if combined_ds.data_vars[var].ndim == 3:
        print(f"Normalising variable {var} with shape {combined_ds.data_vars[var].shape}")
        # Normalise each column separately
        for col in range(combined_ds.data_vars[var].shape[1]):
            min_val = combined_ds[var][:, col, :].min().item()
            max_val = combined_ds[var][:, col, :].max().item()
            mean_val = combined_ds[var][:, col, :].mean().item()
            normalised_stats[f"{var}_col{col}"] = {
                'min': min_val,
                'max': max_val,
                'mean': mean_val}
            normalised_ds[var][:, col, :] = (combined_ds[var][:, col, :] - mean_val) / (max_val - min_val)
    else:        
        min_val = combined_ds[var].min().item()
        max_val = combined_ds[var].max().item()
        mean_val = combined_ds[var].mean().item()
        normalised_stats[var] = {
            'min': min_val,
            'max': max_val,
            'mean': mean_val}
        normalised_ds[var] = (combined_ds[var] - mean_val) / (max_val - min_val)
    

Normalising variable state_t with shape (43, 60, 384)
Normalising variable state_q0001 with shape (43, 60, 384)


In [43]:
print(normalised_ds['state_t'][:, 0, :].min().item())
print(combined_ds['state_t'][:, 0, :].min().item())


-0.6367334962327718
-0.6367334962327718


In [48]:
def normalize_dataset(ds: xr.Dataset) -> (xr.Dataset, dict):
    """
    Normalizes variables in an xarray.Dataset.
    
    3D variables (containing 'lev') are normalized for each level independently 
    across the time and ncol dimensions.
    2D variables are normalized globally across the time and ncol dimensions.
    
    Normalization: (x - x_mean) / (x_max - x_min)
    
    Args:
        ds: The input xarray.Dataset.
        
    Returns:
        The normalized xarray.Dataset.
        A dictionary containing normalization statistics for each variable.
    """
    
    normalized_data = {}
    normalized_stats = {}
    for var_name, da in ds.data_vars.items():
        # Determine the dimensions over which to calculate statistics
        # We always want to normalize across time and ncol.
        stats_dims = ['time', 'ncol']
        
        # Check for 'lev' dimension to determine level-wise normalization
        if 'lev' in da.dims:
            # 3D Variable: Normalization is calculated per level.
            # We calculate mean/min/max over 'time' and 'ncol', preserving 'lev'.
            print(f"Normalizing 3D variable '{var_name}' level-wise...")
            
            x_mean = da.mean(dim=stats_dims) # Resulting dimensions: ('lev',)
            x_min = da.min(dim=stats_dims)   # Resulting dimensions: ('lev',)
            x_max = da.max(dim=stats_dims)   # Resulting dimensions: ('lev',)
            
        else:
            # 2D Variable: Normalization is global.
            # Calculating mean/min/max over 'time' and 'ncol' results in a scalar.
            print(f"Normalizing 2D variable '{var_name}' globally...")
            
            x_mean = da.mean(dim=stats_dims) # Resulting dimensions: scalar
            x_min = da.min(dim=stats_dims)   # Resulting dimensions: scalar
            x_max = da.max(dim=stats_dims)   # Resulting dimensions: scalar


        normalized_stats[var_name] = {
            'mean': x_mean,
            'min': x_min,
            'max': x_max
        }

        # Calculate the range (max - min)
        x_range = x_max - x_min

        # Handle potential division by zero (e.g., if the data is constant)
        # If the range is 0, the normalized result should be 0.
        # This uses xr.where() for efficient conditional operation.
        normalized_da = xr.where(
            x_range == 0,
            0,
            (da - x_mean) / x_range
        )
        
        # Preserve original attributes and add a note about normalization
        normalized_da.attrs = da.attrs.copy()
        if 'units' in normalized_da.attrs:
             # Unitless after normalization
            normalized_da.attrs['original_units'] = normalized_da.attrs.pop('units')
        normalized_da.attrs['normalization'] = 'MinMax scaled: (x - mean) / range'
        
        normalized_data[var_name] = normalized_da
        
    # Create the new Dataset from the normalized DataArrays
    normalized_ds = xr.Dataset(
        normalized_data, 
        coords=ds.coords, 
        attrs=ds.attrs
    )
    
    return normalized_ds, normalized_stats


In [50]:
normalised_ds, normalised_stats = normalize_dataset(combined_ds)

Normalizing 3D variable 'state_t' level-wise...
Normalizing 3D variable 'state_q0001' level-wise...
Normalizing 2D variable 'state_ps' globally...
Normalizing 2D variable 'pbuf_SOLIN' globally...
Normalizing 2D variable 'pbuf_LHFLX' globally...
Normalizing 2D variable 'pbuf_SHFLX' globally...


In [54]:
print(normalised_ds['state_t'][:, 0, :].min().item())
print(combined_ds['state_t'][:, 0, :].min().item())
print(normalised_stats['state_ps'])


-0.5982017293559286
161.56224699763794
{'mean': <xarray.DataArray 'state_ps' ()> Size: 8B
array(98597.14002931), 'min': <xarray.DataArray 'state_ps' ()> Size: 8B
array(73075.29691423), 'max': <xarray.DataArray 'state_ps' ()> Size: 8B
array(103541.62973057)}
